# MDSR — Inference & Evaluation

Loads a trained checkpoint, runs super-resolution inference on validation patches, visualizes reconstructed channels against ground truth, and computes PSNR/SSIM.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

import model_utils.MDSR as mdsr
from utils import Vectorizedtools as vt
from utils import VectorizedInputPreprocessing as vip
from config import Config
from dataset import FreqnetTrainDataLMDB

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## Load normalization stats and a batch of samples

In [ ]:
dataset = FreqnetTrainDataLMDB(Config.LMDB_PATH)
mean = torch.from_numpy(dataset.mean).to(device)
std = torch.from_numpy(dataset.std).to(device)
print(f'Dataset size: {len(dataset)}')

# Pull a small evaluation batch
N_EVAL = 10
batch = [dataset[i] for i in range(N_EVAL)]
PATCHES = torch.stack([b[0] for b in batch])
X       = torch.stack([b[1] for b in batch])
CACHED  = torch.stack([b[2] for b in batch])
Y       = torch.stack([b[3] for b in batch])
print(PATCHES.shape, X.shape, CACHED.shape, Y.shape)

## Load trained checkpoint

In [ ]:
import os
checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, Config.best_checkpoint_name)

model = mdsr.MDSR(
    in_channels=Config.in_channels,
    out_channels=Config.out_channels,
    repeats=Config.repeats,
    layers_per_repeat=Config.layers_per_repeat,
).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded epoch {checkpoint['epoch']} (loss={checkpoint['loss']:.6f})")

## Run inference and reconstruct enhanced image patches

In [ ]:
with torch.no_grad():
    out = model(X.to(device))

denorm = vt.denormalize_zscore(out, mean, std)
out_vector = vip.grid_to_vector(denorm, channel_last=False)
enhanced_dct = vt.vector_to_dctmap(out_vector, CACHED.to(device), height=32, width=32)
EnhancedImage = vt.vectorized_idct(enhanced_dct.detach().cpu().numpy(), is_normalized='none')
print('Enhanced patches:', EnhancedImage.shape)

## Visual comparison: bicubic input vs. ground truth vs. prediction

In [ ]:
def plot_patches(spatial_tensor, title=''):
    sp = spatial_tensor
    if hasattr(sp, 'cpu'):
        sp = sp.cpu().detach().numpy()
    sp = np.asarray(sp)
    cols = 4
    rows = (sp.shape[0] + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8, 2 * rows))
    fig.suptitle(title)
    for i, ax in enumerate(axes.flatten()):
        if i < sp.shape[0]:
            im = sp[i]
            imn = (im - im.min()) / (im.max() - im.min() + 1e-8)
            ax.imshow(imn, cmap='gray', vmin=0, vmax=1)
            ax.set_title(f'chan {i}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

SAMPLE = 0
gt_patches = vt.vectorized_idct(Y[SAMPLE], is_normalized='none')
plot_patches(PATCHES[SAMPLE], 'Bicubic-upsampled input')
plot_patches(gt_patches, 'Ground truth')
plot_patches(EnhancedImage[SAMPLE], 'MDSR prediction')

## Quantitative evaluation — PSNR / SSIM

In [ ]:
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

psnr_metric = PeakSignalNoiseRatio(data_range=1.0)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0)

psnr_model, psnr_baseline = [], []
for i in range(N_EVAL):
    gt = torch.from_numpy(vt.vectorized_idct(Y[i], is_normalized='none')).float()
    pred = torch.from_numpy(EnhancedImage[i]).float()
    base = PATCHES[i].float()
    psnr_model.append(psnr_metric(pred, gt).item())
    psnr_baseline.append(psnr_metric(base, gt).item())

print(f'Mean PSNR — MDSR     : {np.mean(psnr_model):.2f} dB')
print(f'Mean PSNR — bicubic  : {np.mean(psnr_baseline):.2f} dB')
print(f'Gain                 : {np.mean(psnr_model) - np.mean(psnr_baseline):+.2f} dB')

> **Note:** results above are on a small sample batch. For reportable numbers, run over the full held-out validation split and on standard benchmarks (Set5/Set14).